# 01 — Bus inventory

**What is on this bus, how often, and where does it vary?**

Start every new session here. The first useful result is negative: most bytes
on this bus never change, and a byte that never changes is a constant, not a
signal. Ruling those out is what makes the rest tractable.

In [ ]:
import sys
from pathlib import Path

# The project is not installed as a package, so put the repo root on the path.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

from bikecan import dbc, discover, experiment, ids, session

SESSIONS = REPO / "data" / "sessions"

In [ ]:
# Load a session. `legacy` is the 12 unique recordings made on 4 September 2026,
# before this tooling existed -- loose candump logs with no marks and no metadata.
s = session.load(SESSIONS / "legacy")
print(s.summary())

## Every identifier on the bus

`period_ms` is the median inter-arrival gap and is the figure to trust.
`rate_hz` is count over the whole span, so it under-reports badly when the
input spans several recordings with gaps between them.

In [ ]:
inventory = discover.inventory(s.can)
inventory[[
    "id_hex", "count", "period_ms", "jitter_ms", "periodic", "dlc",
    "distinct_payloads", "changing_bytes", "changing_bits", "max_byte_entropy",
]]

## Where the information is

Sort by how much a message actually varies. Anything with one distinct payload
carries no information at all, however often it is sent.

In [ ]:
carries_data = inventory[inventory["distinct_payloads"] > 1]
print(f"{len(carries_data)} of {len(inventory)} identifiers carry any variation at all")
print(f"{inventory['count'].sum():,} frames total, of which "
      f"{carries_data['count'].sum():,} are on a varying identifier")
carries_data[["id_hex", "count", "distinct_payloads", "changing_bytes", "max_byte_entropy"]]

## The dense one, byte by byte

In the legacy corpus this is `04FF3400`: 77 frames, 77 distinct payloads.
`byte_detail` says which bytes hold the variation; the rest are constants to
be ignored.

In [ ]:
target = carries_data.iloc[0]["id_hex"] if len(carries_data) else inventory.iloc[0]["id_hex"]
print(target)
discover.byte_detail(s.can, target)

## Candidate fields

Adjacent changing bits are far more likely to be one multi-bit value than
several unrelated flags, so these runs are where signal fitting starts.

In [ ]:
discover.candidate_fields(s.can, target)

## What the current DBC already decodes

Everything still undecoded is the remaining work, and that list is the project's
real progress metric.

In [ ]:
decoded = dbc.decode_session(s.can)
print(f"{decoded['known'].sum():,} of {len(decoded):,} frames have a message definition")
unknown = sorted(set(decoded.loc[~decoded["known"], "id_hex"]))
print(f"{len(unknown)} identifiers still undecoded:")
print("  " + " ".join(unknown))